1.Data Ingestion

In [1]:
import os
import zipfile
import gdown

In [2]:
SOURCE_URL = "https://drive.google.com/file/d/1z0mreUtRmR-P-magILsDR3T7M6IkGXtY/view?usp=sharing"
LOCAL_DATA_FILE = "artifacts/data_ingestion/data.zip"
UNZIP_DIR = "artifacts/data_ingestion/"

In [3]:
os.chdir("../")
os.makedirs("artifacts/data_ingestion", exist_ok=True)
print(f" Directory created: artifacts/data_ingestion")

 Directory created: artifacts/data_ingestion


In [4]:
file_id = SOURCE_URL.split("/d/")[1].split("/")[0]
download_url = f"https://drive.google.com/uc?id={file_id}"

In [5]:
gdown.download(download_url, LOCAL_DATA_FILE, quiet=False)

print(f"Downloaded to: {LOCAL_DATA_FILE}")

Downloading...
From (original): https://drive.google.com/uc?id=1z0mreUtRmR-P-magILsDR3T7M6IkGXtY
From (redirected): https://drive.google.com/uc?id=1z0mreUtRmR-P-magILsDR3T7M6IkGXtY&confirm=t&uuid=8232ebaf-74f6-4422-b13c-dc5bb2f1a903
To: /artifacts/data_ingestion/data.zip
100%|██████████| 49.0M/49.0M [00:01<00:00, 44.6MB/s]

Downloaded to: artifacts/data_ingestion/data.zip


In [6]:
print(f" Extracting {LOCAL_DATA_FILE} ...")
with zipfile.ZipFile(LOCAL_DATA_FILE, "r") as zip_ref:
    zip_ref.extractall(UNZIP_DIR)
print(f" Extracted to: {UNZIP_DIR}")


 Extracting artifacts/data_ingestion/data.zip ...
 Extracted to: artifacts/data_ingestion/


In [7]:
import os

print("CURRENT DIR:", os.getcwd())
print("FILES HERE:", os.listdir("artifacts/data_ingestion"))

CURRENT DIR: /
FILES HERE: ['Chest-CT-Scan-data', 'data.zip']


In [8]:
%pwd

'/'

02_prepare_base_model

In [9]:
import tensorflow as tf


In [10]:
ROOT_DIR         = "artifacts/prepare_base_model"
BASE_MODEL_PATH  = "artifacts/prepare_base_model/base_model.h5"
UPDATED_MODEL_PATH = "artifacts/prepare_base_model/base_model_updated.h5"

In [11]:

IMAGE_SIZE    = [224, 224, 3]
LEARNING_RATE = 0.01
# Remove the last classification part of the model.
INCLUDE_TOP   = False
# Without ImageNet:
# Your CNN starts with random weights → learns slowly
WEIGHTS       = "imagenet"
CLASSES       = 2

In [12]:
os.makedirs(ROOT_DIR, exist_ok=True)
print(f"✅ Directory created: {ROOT_DIR}")


✅ Directory created: artifacts/prepare_base_model


In [13]:
model = tf.keras.applications.VGG16(
    input_shape=IMAGE_SIZE,
    weights=WEIGHTS,
    include_top=INCLUDE_TOP
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [14]:
import os
%cd ..
base = "artifacts/data_ingestion/Chest-CT-Scan-data"

folders = [
    os.path.join(base, "adenocarcinoma"),
    os.path.join(base, "normal")
]

images = []

for folder in folders:
    images += [
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))
    ]

print("Total images:", len(images))

/content
Total images: 343


In [15]:
model.save(BASE_MODEL_PATH)
print(f"✅ Base model saved to: {BASE_MODEL_PATH}")


✅ Base model saved to: artifacts/prepare_base_model/base_model.h5


In [16]:
for layer in model.layers:
    layer.trainable = False

In [17]:
flatten = tf.keras.layers.Flatten()(model.output)
output  = tf.keras.layers.Dense(units=CLASSES, activation="softmax")(flatten)

In [18]:
full_model = tf.keras.models.Model(inputs=model.input, outputs=output)


In [19]:
full_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [20]:
full_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │        50,178 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,764,866 (56.32 MB)

 Trainable params: 50,178 (196.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [21]:
full_model.save(UPDATED_MODEL_PATH)
print(f"✅ Updated model saved to: {UPDATED_MODEL_PATH}")

✅ Updated model saved to: artifacts/prepare_base_model/base_model_updated.h5


In [22]:
print(os.listdir("artifacts/prepare_base_model"))

['base_model_updated.h5', 'base_model.h5']


03_model_trainer

In [23]:
ROOT_DIR              = "artifacts/training"
TRAINED_MODEL_PATH    = "artifacts/training/model.h5"
UPDATED_MODEL_PATH    = "artifacts/prepare_base_model/base_model_updated.h5"
TRAINING_DATA         = "artifacts/data_ingestion/Chest-CT-Scan-data"

In [35]:
EPOCHS = 1
# Number of full passes through the entire training dataset
# Each epoch = model sees all training images once and updates weights

BATCH_SIZE = 16
# Number of images processed before updating model weights
# Smaller batch = more stable learning, larger batch = faster training

AUGMENTATION = True
# If True, applies random transformations to training images
# (rotation, zoom, flip, shift) to improve generalization and reduce overfitting

IMAGE_SIZE = [224, 224, 3]
# Target input size for the model:
# 224 x 224 = image height and width (standard for VGG16)
# 3 = number of color channels (RGB images)


In [25]:
os.makedirs(ROOT_DIR, exist_ok=True)
print(f"✅ Directory created: {ROOT_DIR}")

✅ Directory created: artifacts/training


In [26]:
model = tf.keras.models.load_model(UPDATED_MODEL_PATH)
print(f"✅ Model loaded from: {UPDATED_MODEL_PATH}")


✅ Model loaded from: artifacts/prepare_base_model/base_model_updated.h5


In [27]:
datagenerator_kwargs = dict(
    rescale=1./255,
    validation_split=0.20
)


In [28]:
dataflow_kwargs = dict(
    target_size=IMAGE_SIZE[:-1],   # (224, 224)
    batch_size=BATCH_SIZE,
    interpolation="bilinear"
)

In [29]:
valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)


In [30]:
valid_generator = valid_datagen.flow_from_directory(
    directory=TRAINING_DATA,
    subset="validation",
    shuffle=False,
    **dataflow_kwargs
)

Found 68 images belonging to 2 classes.


In [31]:
if AUGMENTATION:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=40,
        horizontal_flip=True,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        **datagenerator_kwargs
    )
else:
    train_datagen = valid_datagen

train_generator = train_datagen.flow_from_directory(
    directory=TRAINING_DATA,
    subset="training",
    shuffle=True,
    **dataflow_kwargs
)

Found 275 images belonging to 2 classes.


In [34]:
steps_per_epoch  = train_generator.samples //BATCH_SIZE
validation_steps = valid_generator.samples // BATCH_SIZE

model = tf.keras.models.load_model(UPDATED_MODEL_PATH)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


model.fit(
    train_generator,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    validation_data=valid_generator
)

model.save(TRAINED_MODEL_PATH)
print(f"✅ Trained model saved to: {TRAINED_MODEL_PATH}")

17/17 ━━━━━━━━━━━━━━━━━━━━ 205s 12s/step - accuracy: 0.5676 - loss: 0.7012 - val_accuracy: 0.9375 - val_loss: 0.4501


✅ Trained model saved to: artifacts/training/model.h5


stage_04_model_evaluation

In [44]:
loss, accuracy = model.evaluate(valid_generator)
print(f" Loss: {loss:.4f}")
print(f" Accuracy: {accuracy:.4f}")


5/5 ━━━━━━━━━━━━━━━━━━━━ 47s 8s/step - accuracy: 0.8971 - loss: 0.4662
 Loss: 0.4662
 Accuracy: 0.8971


In [43]:
scores = {"loss": round(loss, 4), "accuracy": round(accuracy, 4)}


In [42]:
SCORE_PATH         = "scores.json"
import json

with open(SCORE_PATH, "w") as f:
    json.dump(scores, f, indent=4)

print(f" Scores saved to: {SCORE_PATH}")

 Scores saved to: scores.json
